In [1]:
from pathlib import Path
from collections import Counter
import pickle

KITTI_DATAPATH = Path("/OpenPCDet/datasets/kitti")

# 1) Count from OpenPCDet dbinfos (recommended for what training uses)
dbinfo_path = KITTI_DATAPATH / "kitti_dbinfos_train.pkl"
with open(dbinfo_path, "rb") as f:
    dbinfos = pickle.load(f)

db_counts = {cls: len(items) for cls, items in dbinfos.items()}
print("Counts from kitti_dbinfos_train.pkl:")
for k, v in sorted(db_counts.items()):
    print(f"{k}: {v}")

# 2) Count raw KITTI labels (full annotation count for label files present)
label_dir = KITTI_DATAPATH / "training" / "label_2"
label_counts = Counter()
for p in label_dir.glob("*.txt"):
    for line in p.read_text().splitlines():
        cls = line.split()[0]
        if cls != "DontCare":
            label_counts[cls] += 1

print("\nCounts from training/label_2:")
for k, v in sorted(label_counts.items()):
    print(f"{k}: {v}")

Counts from kitti_dbinfos_train.pkl:
Car: 14357
Cyclist: 734
Misc: 337
Pedestrian: 2207
Person_sitting: 56
Tram: 224
Truck: 488
Van: 1297

Counts from training/label_2:
Car: 28742
Cyclist: 1627
Misc: 973
Pedestrian: 4487
Person_sitting: 222
Tram: 511
Truck: 1094
Van: 2914


In [3]:
from pathlib import Path
from collections import Counter
import pickle
import numpy as np

NUSC_DATAPATH = Path("/OpenPCDet/datasets/nuscenes/v1.0-trainval")

# 1) Count from dbinfos (closest to what training GT sampler uses)
dbinfo_candidates = sorted(NUSC_DATAPATH.glob("nuscenes_dbinfos_*.pkl"))
if not dbinfo_candidates:
    raise FileNotFoundError(f"No nuscenes_dbinfos_*.pkl found in {NUSC_DATAPATH}")

dbinfo_path = dbinfo_candidates[0]  # pick one; change if you want a specific file
with open(dbinfo_path, "rb") as f:
    dbinfos = pickle.load(f)

db_counts = {cls: len(items) for cls, items in dbinfos.items()}
print(f"Counts from {dbinfo_path.name}:")
for k, v in sorted(db_counts.items()):
    print(f"{k}: {v}")

# 2) Count from infos (annotation objects in info files)
info_paths = []
for name in ["nuscenes_infos_10sweeps_train.pkl", "nuscenes_infos_train.pkl", "nuscenes_infos_val.pkl"]:
    p = NUSC_DATAPATH / name
    if p.exists():
        info_paths.append(p)

if not info_paths:
    info_paths = sorted(NUSC_DATAPATH.glob("nuscenes_infos*.pkl"))

if not info_paths:
    raise FileNotFoundError(f"No nuscenes_infos*.pkl found in {NUSC_DATAPATH}")

info_counts = Counter()
for p in info_paths:
    with open(p, "rb") as f:
        infos = pickle.load(f)

    for info in infos:
        names = info.get("gt_names", [])
        if isinstance(names, np.ndarray):
            names = names.tolist()
        for cls in names:
            if cls and cls != "ignore":
                info_counts[cls] += 1

print("\nCounts from nuscenes infos:")
for k, v in sorted(info_counts.items()):
    print(f"{k}: {v}")

Counts from nuscenes_dbinfos_10sweeps_withvelo.pkl:
barrier: 107507
bicycle: 8185
bus: 12286
car: 339949
construction_vehicle: 11050
ignore: 26297
motorcycle: 8846
pedestrian: 161928
traffic_cone: 62964
trailer: 19202
truck: 65262

Counts from nuscenes infos:
barrier: 107507
bicycle: 8185
bus: 12286
car: 339949
construction_vehicle: 11050
motorcycle: 8846
pedestrian: 161928
traffic_cone: 62964
trailer: 19202
truck: 65262
